# v34.3 fixed — targeted recheck with nested-schema alignment

This notebook fixes the old v34.1/v34.3 alignment bug. The previous code only indexed dataset records with a flat `question` key, but the current files often store `questions: [...]` with shared `premises-FOL`.

Goal: re-audit the 7 v34 sweep flips `[69,79,132,141,144,146,150]` without any new sweep or model call. It keeps a flip only if it passes: G1 direct universal interrogative, G2 affirmative conclusion only, G3 old Final Answer is No, and G4 Horn proof with contrapositive from aligned `premises-FOL`.


In [ ]:
# ============================================================
# v34.3 FIXED - targeted recheck with nested-schema alignment
# No sweep, no model, no gold in selection.
# Fixes the old v34.1/v34.3 failure mode where dataset records use:
#   questions: [...]
#   answers: [...]
#   premises-FOL: [...]
# instead of a single `question` key.
# ============================================================
import json, re, glob, os, math
from collections import Counter
from pathlib import Path

VERSION = 'v34_3_targeted_recheck_fixed_alignment'
LABELS = ['A','B','C','D','Yes','No','Unknown']
V27_MACRO = 0.5833102471757934
V301_MACRO = 0.5934206145879246
V322_MACRO = 0.596858548901435
V342_MACRO = 0.6098313451254628
V301_DIFFS = {71,109,125}
V322_DIFFS = {14,71,109,125}

CANDIDATE_FLIP_SET = {69,79,132,141,144,146,150}
FALSE_FLIP_KNOWN_FOR_AUDIT_ONLY = 69  # never used in selection except reported posthoc if gold exists in preds

SEARCH_ROOTS = [
    '/kaggle/working',
    '/kaggle/input',
    '/mnt/data',
    '.',
]

def find_file(fname, extra=(), required=True):
    names = [fname] if isinstance(fname, str) else list(fname)
    for h in extra:
        if h and os.path.exists(h):
            return h
    for root in SEARCH_ROOTS:
        for name in names:
            g = glob.glob(f'{root}/**/{name}', recursive=True)
            if g:
                # prefer working/current outputs over stale input copies by short path then mtime
                return sorted(g, key=lambda p: (0 if '/working/' in p else 1, len(p), p))[0]
    if required:
        raise FileNotFoundError(f'FATAL: could not find {names} in {SEARCH_ROOTS}')
    return None

def load_json(name, required=True, extra=()):
    p = find_file(name, extra=extra, required=required)
    if p is None:
        return None, None
    with open(p, 'r', encoding='utf-8') as f:
        return json.load(f), p

def metrics(rows):
    tp, fp, fn = Counter(), Counter(), Counter()
    for r in rows:
        g, p = r.get('gold'), r.get('pred')
        if g == p:
            tp[g] += 1
        else:
            fp[p] += 1
            fn[g] += 1
    per = {}
    for l in LABELS:
        pr = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        rc = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        per[l] = dict(precision=pr, recall=rc, f1=f1, support=tp[l]+fn[l], pred_count=tp[l]+fp[l])
    tot = sum(v['support'] for v in per.values())
    return dict(
        n=len(rows),
        acc=sum(r.get('gold') == r.get('pred') for r in rows) / len(rows),
        macro_f1=sum(v['f1'] for v in per.values()) / len(LABELS),
        weighted_f1=sum(v['f1'] * v['support'] for v in per.values()) / max(1, tot),
        mc_macro=sum(per[l]['f1'] for l in 'ABCD') / 4,
        ynu_macro=sum(per[l]['f1'] for l in ('Yes','No','Unknown')) / 3,
        per_label=per,
        gold_count=Counter(r.get('gold') for r in rows),
        pred_count=Counter(r.get('pred') for r in rows),
    )

def compact_metrics(m):
    return {k:m[k] for k in ('n','acc','macro_f1','weighted_f1','mc_macro','ynu_macro')}

def show(m, name):
    print(f"--- {name} ---")
    print(f"acc={m['acc']:.4f} macro-F1={m['macro_f1']:.10f} weighted-F1={m['weighted_f1']:.4f} "
          f"MC_macro={m['mc_macro']:.4f} YNU_macro={m['ynu_macro']:.4f}")
    for l in LABELS:
        v = m['per_label'][l]
        print(f"  {l:8s} P={v['precision']:.3f} R={v['recall']:.3f} F1={v['f1']:.3f} gold={v['support']} pred={v['pred_count']}")

def diff_indices(a, b):
    return sorted(x.get('idx') for x,y in zip(a,b) if x.get('pred') != y.get('pred'))

# ------------------ load + verify baseline chain ------------------
v27, v27_path = load_json('v27_standard_preds.json', extra=['/kaggle/input/datasets/yixuanisthebest/v2333333/v27_standard_preds.json'])
v301, v301_path = load_json(['v30_1_full_preds.json','v30_full_preds.json'])
try:
    v322, v322_path = load_json(['v32_2_full_preds.json','v32_2_independent_full_preds.json'])
except FileNotFoundError:
    print('v32_2 preds not found -> reconstruct v32.2 = v30.1 + idx14->Yes')
    v322_path = 'reconstructed_from_v30_1'
    v322 = []
    for r in v301:
        rr = dict(r)
        if rr.get('idx') == 14:
            rr['pred'] = 'Yes'
            rr['repair'] = rr.get('repair','') or 'reconstructed_v32_2_idx14'
        v322.append(rr)
v34, v34_path = load_json(['v34_unified_cpu_best_preds.json','v34_unified_final_preds.json'])

m27, m301, m322, m34 = metrics(v27), metrics(v301), metrics(v322), metrics(v34)
assert abs(m27['macro_f1'] - V27_MACRO) < 1e-9, f'v27 mismatch: {m27["macro_f1"]}'
assert abs(m301['macro_f1'] - V301_MACRO) < 1e-9, f'v30.1 mismatch: {m301["macro_f1"]}'
assert abs(m322['macro_f1'] - V322_MACRO) < 1e-9, f'v32.2 mismatch: {m322["macro_f1"]}'
assert set(diff_indices(v27, v322)) == V322_DIFFS, f'v32.2 diffs corrupted: {diff_indices(v27, v322)}'
print('Loaded:')
print('  v27 :', v27_path)
print('  v30 :', v301_path)
print('  v32.2:', v322_path)
print('  v34 :', v34_path)
print(f'v32.2 VERIFIED: macro={m322["macro_f1"]:.10f}, diffs={diff_indices(v27, v322)}')
print(f'v34 reference recomputed: macro={m34["macro_f1"]:.10f}')

cand_flips = []
for a,b in zip(v322, v34):
    if a.get('pred') != b.get('pred'):
        cand_flips.append((a.get('idx'), a.get('pred'), b.get('pred'), dict(a)))
print('Candidate flips from v32.2 to v34:', [(i,o,n) for i,o,n,_ in cand_flips])
assert {i for i,_,_,_ in cand_flips} == CANDIDATE_FLIP_SET, 'unexpected v34 candidate flips; STOP'
assert all(o == 'No' and n == 'Yes' for _,o,n,_ in cand_flips), 'unexpected flip direction; STOP'

# ------------------ robust dataset alignment ------------------
def normq(q):
    return re.sub(r'\s+', ' ', str(q).strip().lower())[:500]

def candidate_records(raw):
    if isinstance(raw, list):
        return raw
    if isinstance(raw, dict):
        for k in ('data','samples','records','items'):
            if isinstance(raw.get(k), list):
                return raw[k]
        # dict keyed by id -> rec
        out=[]
        for k,v in raw.items():
            if isinstance(v, dict):
                vv=dict(v); vv.setdefault('idx', k); out.append(vv)
        return out
    return []

def get_premises_fol(rec):
    for key in ('premises-FOL','premises_FOL','premises_fol','fol_premises'):
        if key in rec:
            v = rec[key]
            return v if isinstance(v, list) else [v]
    for k,v in rec.items():
        lk = str(k).lower()
        if 'fol' in lk and 'premise' in lk:
            return v if isinstance(v, list) else [v]
    for k,v in rec.items():
        if 'fol' in str(k).lower():
            return v if isinstance(v, list) else [v]
    return []

def build_question_index(records):
    byq = {}
    collisions = Counter()
    for rec_i, rec in enumerate(records):
        if not isinstance(rec, dict):
            continue
        # nested generated/data schema
        if isinstance(rec.get('questions'), list):
            for qi, q in enumerate(rec['questions']):
                key = normq(q)
                item = {'record': rec, 'rec_i': rec_i, 'q_i': qi, 'question': q, 'premises_fol': get_premises_fol(rec)}
                if key in byq:
                    collisions[key] += 1
                    # keep list for exact idx_component fallback if needed
                    if isinstance(byq[key], list):
                        byq[key].append(item)
                    else:
                        byq[key] = [byq[key], item]
                else:
                    byq[key] = item
        # flat schema
        for k in ('question','query','prompt'):
            if k in rec and rec[k]:
                key = normq(rec[k])
                item = {'record': rec, 'rec_i': rec_i, 'q_i': None, 'question': rec[k], 'premises_fol': get_premises_fol(rec)}
                if key not in byq:
                    byq[key] = item
                break
    return byq, collisions

data_raw, data_path = load_json('Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json')
records = candidate_records(data_raw)
byq, collisions = build_question_index(records)
print('Dataset:', data_path)
print('Dataset records:', len(records), 'question-index entries:', len(byq), 'colliding question keys:', len(collisions))

# Optional fallback: index full-test generated too, useful if Kaggle only has generated_v4style_300 in input.
gen_path = find_file('generated_v4style_300.json', required=False)
if gen_path:
    gen_raw = json.load(open(gen_path, encoding='utf-8'))
    gen_byq, _ = build_question_index(candidate_records(gen_raw))
    # do not override main data; only fill missing
    add=0
    for k,v in gen_byq.items():
        if k not in byq:
            byq[k]=v; add += 1
    print('Added fallback question-index entries from generated_v4style_300:', add)

def lookup_question(q):
    hit = byq.get(normq(q))
    if isinstance(hit, list):
        return hit[0]
    return hit

# ------------------ Horn/FOL mini-engine ------------------
def norm_fol(s):
    s = str(s)
    replacements = [('∀','ForAll '),('∃','Exists '),('→','->'),('⟶','->'),('∧','&'),('∨','|'),('¬','~'),('↔','<->')]
    for a,b in replacements:
        s = s.replace(a,b)
    for _ in range(10):
        m = re.search(r'Implies\s*\(([^()]*(?:\([^()]*\)[^()]*)*?),\s*([^()]*(?:\([^()]*\)[^()]*)*?)\)', s)
        if not m:
            break
        s = s[:m.start()] + f'({m.group(1)} -> {m.group(2)})' + s[m.end():]
    return s

ATOM = re.compile(r'(~?)\s*([A-Za-z_][A-Za-z0-9_]*)\s*\(([^()]*)\)')
RES = {'ForAll','Exists','Implies','And','Or','Not'}
def atoms_of(s):
    return [(name, neg == '~') for neg, name, _ in ATOM.findall(str(s)) if name not in RES]

def build_kb(pfs):
    kb = {'edges': [], 'ufacts': set(), 'efacts': set(), 'raw': list(pfs or [])}
    for pf in (pfs or []):
        try:
            s = norm_fol(pf)
            quant = 'U' if re.search(r'\bForAll\b', s) or '∀' in str(pf) else ('E' if re.search(r'\bExists\b', s) or '∃' in str(pf) else 'F')
            if '->' in s:
                ant, cons = s.split('->', 1)
                A, C = atoms_of(ant), atoms_of(cons)
                if not A or not C:
                    continue
                antset = frozenset((a, an) for a,an in A)
                for cn,cneg in C:
                    kb['edges'].append((antset, (cn,cneg), quant, str(pf)))
                    # valid contrapositive only for single antecedent universal edges
                    if quant == 'U' and len(A) == 1:
                        a0, an0 = A[0]
                        kb['edges'].append((frozenset([(cn, not cneg)]), (a0, not an0), quant, 'CONTRAPOSITIVE_OF: '+str(pf)))
            else:
                for n,neg in atoms_of(s):
                    if not neg:
                        (kb['ufacts'] if quant == 'U' else kb['efacts']).add(n)
                    elif quant == 'U':
                        kb['ufacts'].add('~'+n)
        except Exception as e:
            pass
    return kb

def u_closure(kb, seeds):
    derived = {(s, False) for s in seeds}
    for f in kb['ufacts']:
        if isinstance(f,str) and f.startswith('~'):
            derived.add((f[1:], True))
        else:
            derived.add((f, False))
    proof = {lit: ('seed' if lit[0] in seeds else 'universal_fact') for lit in derived}
    changed = True
    while changed:
        changed = False
        for ant, cons, q, raw in kb['edges']:
            if q != 'U':
                continue
            if ant <= derived and cons not in derived:
                derived.add(cons)
                proof[cons] = raw
                changed = True
    return derived, proof

def _stem(w):
    return w[:-1] if w.endswith('s') and len(w) > 3 else w

def split_camel(name):
    return set(_stem(w.lower()) for w in re.findall(r'[A-Z][a-z0-9]*|[a-z]+', str(name)) if len(w) > 2)

STOP = set('do does are is the a an all every each according premises premise following statement true based above for that have has had will can must of in to with on it from and or if then implies student students researcher researchers developer developers person people individual individuals'.split())
def words(text):
    text = re.sub(r'(?<=[a-z0-9])(?=[A-Z])', ' ', str(text))
    return set(_stem(w) for w in re.findall(r'[a-z]+', text.lower()) if len(w) > 2 and w not in STOP)

def kb_names(kb):
    names = set()
    for f in kb['ufacts'] | kb['efacts']:
        names.add(f[1:] if isinstance(f,str) and f.startswith('~') else f)
    for ant, (cn,_), _, _ in kb['edges']:
        names.add(cn)
        names |= {n for n,_ in ant}
    return {n for n in names if n}

def match_pred(text_words, kb, used=()):
    best, score = None, 0.0
    for c in kb_names(kb) - set(used):
        cw = split_camel(c)
        if not cw:
            continue
        # Jaccard-ish with recall bias for predicate tokens
        s = len(cw & text_words) / max(1, len(cw))
        if s > score:
            best, score = c, s
    return (best, score) if score >= 0.5 else (None, score)

def g4_horn_proof(question, pfs):
    if not pfs:
        return False, 'G4 no premises-FOL available', {}
    kb = build_kb(pfs)
    ql = question.lower()
    # direct universal interrogative seed: all/every <domain>
    seeds = set()
    domain_text = ''
    md = re.search(r'\b(?:all|every)\s+([a-z]+)', ql)
    if md:
        domain_text = md.group(1)
        C, cscore = match_pred(words(domain_text), kb)
        if C:
            seeds.add(C)
    # target predicate from remaining question words
    target_words = words(ql) - words(domain_text)
    Q, qscore = match_pred(target_words, kb, used=seeds)
    if not Q:
        return False, f'G4 no target predicate match (score={qscore:.2f})', {'seeds': sorted(seeds), 'target_score': qscore, 'kb_names': sorted(kb_names(kb))[:30]}
    closure, proof = u_closure(kb, seeds)
    ok = (Q, False) in closure
    cert = {
        'target': Q,
        'target_score': qscore,
        'seeds': sorted(seeds),
        'closure_positive': sorted(n for n,neg in closure if not neg),
        'closure_negative': sorted(n for n,neg in closure if neg),
        'proof_edge': proof.get((Q,False)),
        'premises_fol_used': pfs,
    }
    return ok, (f'G4 PROVEN {Q} from seeds={sorted(seeds)}' if ok else f'G4 NOT_DERIVABLE {Q} from seeds={sorted(seeds)}'), cert

# ------------------ Guards G1-G3 ------------------
def g123(row):
    q = str(row.get('question','')).strip()
    e = str(row.get('explanation',''))
    ql = q.lower()
    if not re.match(r'^(do|are|does|is)\b', ql):
        return False, 'G1 not_interrogative', {}
    if not re.search(r'\b(all|every)\b', ql):
        return False, 'G1 not_universal', {}
    if re.search(r'statement\s*:', ql):
        return False, 'G1 statement_form', {}
    if re.search(r'\bif\b|\bimplies\b|\bthen\b', ql):
        return False, 'G1 conditional_in_question', {}
    body = re.split(r'Final Answer', e, flags=re.I)[0]
    # Conclusion sentence only. Negation earlier in body is allowed for modus tollens.
    matches = list(re.finditer(r'\b(?:Therefore|Thus|Hence)\b[^.\n]{0,240}', body, re.I))
    if not matches:
        return False, 'G2 no_conclusion_marker', {'body_tail': body[-500:]}
    concl = matches[-1].group(0)
    if 'all' not in concl.lower() and 'every' not in concl.lower():
        return False, 'G2 conclusion_not_universal', {'conclusion': concl}
    # Only block if the conclusion itself is negative. Do NOT block negation elsewhere in reasoning.
    if re.search(r'\b(no|not|cannot|never|without|fail|fails|lack|lacks|unable)\b', concl, re.I):
        return False, 'G2 negated_conclusion', {'conclusion': concl}
    final = re.findall(r'Final Answer\s*[:\-]?\s*(Yes|No|Unknown|[ABCD])\b', e, re.I)
    if not final or final[-1].title() != 'No':
        return False, 'G3 final_not_No', {'finals': final[-3:]}
    return True, 'G1-G3 PASS', {'conclusion': concl, 'finals': final[-3:]}

# ------------------ Apply targeted audit ------------------
kept, removed, certificates = [], [], []
for idx, old, newp, base_row in cand_flips:
    cert = {
        'idx': idx,
        'question': base_row.get('question'),
        'old_pred': old,
        'v34_pred': newp,
        'gold_for_audit_only': base_row.get('gold'),
    }
    ok123, reason123, cert123 = g123(base_row)
    cert.update(cert123)
    if not ok123:
        cert.update({'kept': False, 'reason': reason123, 'dataset_aligned': False})
        removed.append((idx, reason123))
        certificates.append(cert)
        print(f'idx={idx:3d} REMOVED | {reason123}')
        continue
    hit = lookup_question(base_row.get('question',''))
    if hit is None:
        cert.update({'kept': False, 'reason': 'G4 no dataset alignment', 'dataset_aligned': False})
        removed.append((idx, 'G4 no dataset alignment'))
        certificates.append(cert)
        print(f'idx={idx:3d} REMOVED | G4 no dataset alignment')
        continue
    pfs = hit.get('premises_fol') or []
    ok4, reason4, cert4 = g4_horn_proof(base_row.get('question',''), pfs)
    cert.update({'dataset_aligned': True, 'dataset_rec_i': hit.get('rec_i'), 'dataset_q_i': hit.get('q_i'), 'horn_certificate': cert4})
    if ok4:
        kept.append((idx, reason4))
        cert.update({'kept': True, 'reason': reason4})
        print(f'idx={idx:3d} KEPT    | {reason4}')
    else:
        removed.append((idx, reason4))
        cert.update({'kept': False, 'reason': reason4})
        print(f'idx={idx:3d} REMOVED | {reason4}')
    certificates.append(cert)

kept_ids = {i for i,_ in kept}
new_rows = []
for r in v322:
    rr = dict(r)
    if rr.get('idx') in kept_ids and rr.get('pred') == 'No':
        rr['pred'] = 'Yes'
        rr['repair'] = 'v34_3_fixed_alignment_targeted_horn_recheck'
    new_rows.append(rr)

nm = metrics(new_rows)
show(m322, 'BASE v32.2')
show(nm, VERSION)
show(m34, 'v34 sweep reference')

# gates: selection is against v32.2, but we report vs v34/v34.2 too.
gain_vs_v322 = nm['macro_f1'] - m322['macro_f1']
gain_vs_v342 = nm['macro_f1'] - V342_MACRO
passes_gate = (
    gain_vs_v322 > 0.01 and
    abs(nm['mc_macro'] - m322['mc_macro']) < 1e-12 and
    nm['per_label']['Unknown']['f1'] >= m322['per_label']['Unknown']['f1'] - 1e-12 and
    nm['per_label']['No']['f1'] >= m322['per_label']['No']['f1'] - 0.05 and
    not any(v['pred_count'] == 0 and v['support'] > 0 for v in nm['per_label'].values())
)
status = 'SELECT_V34_3_FIXED' if passes_gate else 'ROLLBACK_v32_2'
selected_rows = new_rows if passes_gate else v322
selected_metrics = nm if passes_gate else m322
expected_diffs_vs_v27 = sorted(V322_DIFFS | kept_ids) if passes_gate else sorted(V322_DIFFS)

print('\nDECISION:', status)
print(f'gain vs v32.2 = {gain_vs_v322:+.10f}')
print(f'gain vs v34.2 = {gain_vs_v342:+.10f}')
print(f'v34 sweep reference macro = {m34["macro_f1"]:.10f}')
print('kept:', kept)
print('removed:', removed)

risk_report = {
    'version': VERSION + '_risk_report',
    'final_decision': status,
    'artifact_risk': 'LOW_RELOADED_SAVED_PREDS',
    'overfit_risk': 'LOWER_THAN_V34_NO_NEW_SWEEP_TARGETED_PROOF_FILTER',
    'rule_provenance': 'Fixed semantic guards over the 7 v34 proposed flips only; nested-schema premises-FOL alignment; Horn proof with contrapositive; no broad sweep.',
    'label_collapse': False,
    'underfit_risk': 'STILL_PRESENT_MC_UNTOUCHED_AND_RESIDUAL_YNU',
    'kept_flips': [i for i,_ in kept],
    'removed_flips': [list(x) for x in removed],
    'metrics': compact_metrics(selected_metrics),
    'candidate_metrics_before_selection': compact_metrics(nm),
    'delta_vs_v32_2': {
        'acc': selected_metrics['acc'] - m322['acc'],
        'macro_f1': selected_metrics['macro_f1'] - m322['macro_f1'],
        'weighted_f1': selected_metrics['weighted_f1'] - m322['weighted_f1'],
        'mc_macro': selected_metrics['mc_macro'] - m322['mc_macro'],
        'ynu_macro': selected_metrics['ynu_macro'] - m322['ynu_macro'],
    },
    'delta_candidate_vs_v34_2': {
        'macro_f1': nm['macro_f1'] - V342_MACRO,
    },
    'v34_reference_macro_f1': m34['macro_f1'],
}

summary = {
    'version': VERSION,
    'selection_status': status,
    'selected_source': 'v32.2 + v34.3 fixed targeted Horn flips' if passes_gate else 'ROLLBACK_v32_2',
    'rule': 'G1 direct universal interrogative + G2 affirmative conclusion only + G3 old Final=No + G4 Horn proof with contrapositive; nested questions[]/premises-FOL alignment fixed',
    'loaded_paths': {
        'v27': v27_path,
        'v30_1': v301_path,
        'v32_2': v322_path,
        'v34': v34_path,
        'dataset': data_path,
    },
    'kept_flips': [i for i,_ in kept],
    'removed_flips': [list(x) for x in removed],
    'certificates': certificates,
    'candidate_metrics': nm,
    'selected_metrics': selected_metrics,
    'v32_2_baseline': m322,
    'v34_2_reference_macro_f1': V342_MACRO,
    'v34_sweep_reference': m34,
    'risk_report': risk_report,
}

outdir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
outdir.mkdir(parents=True, exist_ok=True)
for tag in ('v34_3_standard_fixed','v34_3_full_fixed'):
    pred_path = outdir / f'{tag}_preds.json'
    summary_path = outdir / f'{tag}_summary.json'
    with open(pred_path, 'w', encoding='utf-8') as f:
        json.dump(selected_rows, f, ensure_ascii=False, indent=1)
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=1)
    # reload verify
    with open(pred_path, encoding='utf-8') as f:
        rp = json.load(f)
    rm = metrics(rp)
    rd = diff_indices(v27, rp)
    assert abs(rm['macro_f1'] - selected_metrics['macro_f1']) < 1e-12, f'{tag} metric reload mismatch'
    assert set(rd) == set(expected_diffs_vs_v27), f'{tag} diffs mismatch: {rd} != {expected_diffs_vs_v27}'
    print(f'SAVED+VERIFIED {pred_path.name} / {summary_path.name} diffs_vs_v27={rd}')

risk_path = outdir / 'v34_3_risk_report_fixed.json'
with open(risk_path, 'w', encoding='utf-8') as f:
    json.dump(risk_report, f, ensure_ascii=False, indent=1)
print('SAVED risk report:', risk_path)
print('ALL ASSERTS PASSED')
